In [15]:
#Importing Necessary libraries
import pandas as pd
import sqlite3

In [17]:
#Loading the dataset
df = pd.read_csv("Telco-Customer-Churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [18]:
#Understanding the dataset
print("Shape:", df.shape)
print("\nColumn Names:\n", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())

Shape: (7043, 21)

Column Names:
 ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data Types:
 customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Missing Values:
 customer

In [19]:
#Basic statistics
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [20]:
#Fix TotalCharges from string to number
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

#Fix SeniorCitizen from 0/1 to Yes/No
df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})

print("Fixed! TotalCharges dtype:", df['TotalCharges'].dtype)
print("SeniorCitizen values:", df['SeniorCitizen'].unique())

Fixed! TotalCharges dtype: float64
SeniorCitizen values: ['No' 'Yes']


In [21]:
#How many customers churned?
print(df['Churn'].value_counts())
print("\nChurn Rate:", round(df['Churn'].value_counts(normalize=True)['Yes'] * 100, 2), "%")

Churn
No     5174
Yes    1869
Name: count, dtype: int64

Churn Rate: 26.54 %


In [22]:
#Setting up SQLite database
conn = sqlite3.connect("churn.db")
df.to_sql("customers", conn, if_exists="replace", index=False)
print("Database created successfully!")

Database created successfully!


In [23]:
#SQL Query 1: Total customers and churn count
query = """
SELECT 
    Churn,
    COUNT(*) as total_customers,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM customers), 2) as percentage
FROM customers
GROUP BY Churn
"""
pd.read_sql_query(query, conn)

,Churn,total_customers,percentage
0,No,5174,73.46
1,Yes,1869,26.54


In [24]:
# SQL Query 2: Churn by contract type
query2 = """
SELECT 
    Contract,
    Churn,
    COUNT(*) as total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY Contract), 2) as churn_rate
FROM customers
GROUP BY Contract, Churn
ORDER BY Contract, Churn
"""
pd.read_sql_query(query2, conn)

,Contract,Churn,total,churn_rate
0,Month-to-month,No,2220,57.29
1,Month-to-month,Yes,1655,42.71
2,One year,No,1307,88.73
3,One year,Yes,166,11.27
4,Two year,No,1647,97.17
5,Two year,Yes,48,2.83


In [25]:
# SQL Query 3: Churn by tenure group
query3 = """
SELECT 
    CASE 
        WHEN tenure <= 12 THEN '0-12 months'
        WHEN tenure <= 24 THEN '13-24 months'
        WHEN tenure <= 48 THEN '25-48 months'
        ELSE '49+ months'
    END as tenure_group,
    Churn,
    COUNT(*) as total
FROM customers
GROUP BY tenure_group, Churn
ORDER BY tenure_group, Churn
"""
pd.read_sql_query(query3, conn)

,tenure_group,Churn,total
0,0-12 months,No,1149
1,0-12 months,Yes,1037
2,13-24 months,No,730
3,13-24 months,Yes,294
4,25-48 months,No,1269
5,25-48 months,Yes,325
6,49+ months,No,2026
7,49+ months,Yes,213
